In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

source_path = "abfss://source@storegare.dfs.core.windows.net/sales/"
schema_loc  = "abfss://source@storegare.dfs.core.windows.net/_autoloader/schema/bronze_sales/"
ckpt_loc    = "abfss://source@storegare.dfs.core.windows.net/_autoloader/checkpoints/bronze_sales/"

df3 = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.schemaLocation", schema_loc)   # REQUIRED for Auto Loader
      .option("rescuedDataColumn", "_rescued_data")       # keeps unexpected fields safely
      .load(source_path)
)

# ✅ add ingestion metadata
df3 = (df3
      .withColumn("ingestion_time", current_timestamp())
      .withColumn("source_file", input_file_name())
)

query = (df3.writeStream
         .format("delta")
         .option("checkpointLocation", ckpt_loc)
         .outputMode("append")
         .trigger(processingTime="1 minute")  # keeps running and checking for new files
         .table("databricks_7405611871831248.bronze.sales_raw"))

In [0]:
df3=spark.read.table("databricks_7405611871831248.bronze.sales_raw")
display(df3)

In [0]:
query.stop()# stoping the while loop. 

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
import time

source_path = "abfss://source@storegare.dfs.core.windows.net/sales/"

run_id = str(int(time.time()))  # makes a new folder name every time you run

schema_loc  = f"abfss://source@storegare.dfs.core.windows.net/_autoloader/schema/bronze_sales_{run_id}/"
ckpt_loc    = f"abfss://source@storegare.dfs.core.windows.net/_autoloader/checkpoints/bronze_sales_{run_id}/"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.schemaLocation", schema_loc)
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # evolution ON
      .option("rescuedDataColumn", "_rescued_data")
      .load(source_path)
      .withColumn("ingestion_time", current_timestamp())
      .withColumn("source_file", input_file_name())
)

(df.writeStream
 .format("delta")
 .option("checkpointLocation", ckpt_loc)
 .outputMode("append")
 .trigger(availableNow=True)   # ✅ run once and stop (best for testing)
 .table("databricks_7405611871831248.bronze.sales_raw_evolving"))

In [0]:
df3=spark.read.table("databricks_7405611871831248.bronze.sales_raw_evolving")
display(df3)

# for the silver layer

In [0]:
from pyspark.sql.functions import col, trim, upper, concat_ws

BRONZE = "databricks_7405611871831248.bronze.sales_raw_evolving"
SILVER = "databricks_7405611871831248.silver.fires_clean"

df = spark.table(BRONZE)

silver_df = (
    df
    .withColumn("FIRE_NAME", upper(trim(col("FIRE_NAME"))))
    .withColumn("STATE", upper(trim(col("STATE"))))
    .withColumn("FIRE_SIZE", col("FIRE_SIZE").cast("double"))
    .withColumn("LATITUDE", col("LATITUDE").cast("double"))
    .withColumn("LONGITUDE", col("LONGITUDE").cast("double"))
    .withColumn("FIRE_YEAR", col("FIRE_YEAR").cast("int"))
    .filter(
        col("STATE").isNotNull() &
        col("LATITUDE").between(-90, 90) &
        col("LONGITUDE").between(-180, 180) &
        col("FIRE_YEAR").between(1900, 2100) &
        col("FIRE_NAME").isNotNull()
    )
    .withColumn("fire_key", concat_ws("||",
        col("FIRE_NAME"),
        col("FIRE_YEAR").cast("string"),
        col("LATITUDE").cast("string"),
        col("LONGITUDE").cast("string")
    ))
    .select(
        "fire_key",
        "FIRE_NAME","FIRE_SIZE","STATE","LATITUDE","LONGITUDE","FIRE_YEAR",
        "ingestion_time","source_file"
    )
)

(silver_df.write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema", "true").saveAsTable(SILVER))

In [0]:
%sql
SELECT * FROM databricks_7405611871831248.silver.fires_clean LIMIT 20;

# bad record to corontine

In [0]:
BAD_TABLE = "databricks_7405611871831248.silver.fires_bad"

silver_df_bad = silver_df.filter(
    col("LATITUDE").isNull() |
    col("LONGITUDE").isNull() |
    col("FIRE_YEAR").isNull()
)

(silver_df_bad.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(BAD_TABLE))

# merging the data with new record

In [0]:
%sql SHOW TABLES IN databricks_7405611871831248.bronze

# creating staging. with id and  as one view for the join

In [0]:
from pyspark.sql.functions import sha2, concat_ws, row_number, desc
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper

BRONZE_TABLE = "databricks_7405611871831248.bronze.sales_raw_evolving"

bronze_df = spark.table(BRONZE_TABLE)

staging = (
    bronze_df
    .withColumn("FIRE_NAME", upper(trim(col("FIRE_NAME"))))
    .withColumn("STATE", upper(trim(col("STATE"))))
    .withColumn("FIRE_SIZE", col("FIRE_SIZE").cast("double"))
    .withColumn("LATITUDE", col("LATITUDE").cast("double"))
    .withColumn("LONGITUDE", col("LONGITUDE").cast("double"))
    .withColumn("FIRE_YEAR", col("FIRE_YEAR").cast("int"))
    .filter(
        col("STATE").isNotNull() &
        col("LATITUDE").between(-90, 90) &
        col("LONGITUDE").between(-180, 180) &
        col("FIRE_YEAR").between(1900, 2100) &
        col("FIRE_NAME").isNotNull()
    )
    .withColumn(
        "fire_key",
        sha2(concat_ws("||",
            col("FIRE_NAME"),
            col("FIRE_YEAR").cast("string"),
            col("LATITUDE").cast("string"),
            col("LONGITUDE").cast("string")
        ), 256)
    )
    .select(
        "fire_key",
        "FIRE_NAME","FIRE_SIZE","STATE","LATITUDE","LONGITUDE","FIRE_YEAR",
        "ingestion_time","source_file"
    )
)

w = Window.partitionBy("fire_key").orderBy(desc("ingestion_time"))
staging_dedup = (staging
                 .withColumn("rn", row_number().over(w))
                 .filter(col("rn") == 1)
                 .drop("rn"))

staging_dedup.createOrReplaceTempView("fires_staging")

# Step 5 — Merging---with old silver


In [0]:
%sql
SELECT COUNT(*) FROM databricks_7405611871831248.silver.fires_clean;

SELECT * FROM databricks_7405611871831248.silver.fires_clean
ORDER BY ingestion_time DESC
LIMIT 20;

In [0]:
spark.sql("""MERGE INTO databricks_7405611871831248.silver.fires_clean AS t USING fires_staging s
ON t.fire_key = s.fire_key
WHEN MATCHED AND s.ingestion_time > t.ingestion_time THEN
  UPDATE SET
    t.FIRE_NAME = s.FIRE_NAME,
    t.STATE = s.STATE,
    t.FIRE_SIZE = s.FIRE_SIZE,
    t.LATITUDE = s.LATITUDE,
    t.LONGITUDE = s.LONGITUDE,
    t.FIRE_YEAR = s.FIRE_YEAR,
    t.ingestion_time = s.ingestion_time,
    t.source_file = s.source_file
WHEN NOT MATCHED THEN
  INSERT (
    fire_key, FIRE_NAME, FIRE_SIZE, STATE, LATITUDE, LONGITUDE, FIRE_YEAR, ingestion_time, source_file
  )
  VALUES (
    s.fire_key, s.FIRE_NAME, s.FIRE_SIZE, s.STATE, s.LATITUDE, s.LONGITUDE, s.FIRE_YEAR, s.ingestion_time, s.source_file
  );""")



# gold layer 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_7405611871831248.gold.fire_count_state_year
USING DELTA
AS
SELECT
  STATE,
  FIRE_YEAR,
  COUNT(*) AS fire_count,
  MAX(ingestion_time) AS last_ingested_time
FROM databricks_7405611871831248.silver.fires_clean
GROUP BY STATE, FIRE_YEAR;

## check gold

In [0]:
%sql
SELECT * 
FROM databricks_7405611871831248.gold.fire_count_state_year
ORDER BY FIRE_YEAR DESC, fire_count DESC
LIMIT 50;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_7405611871831248.gold.fire_size_state_year
USING DELTA
AS
SELECT
  STATE,
  FIRE_YEAR,
  SUM(FIRE_SIZE) AS total_fire_size,
  AVG(FIRE_SIZE) AS avg_fire_size,
  MAX(FIRE_SIZE) AS max_fire_size,
  MAX(ingestion_time) AS last_ingested_time
FROM databricks_7405611871831248.silver.fires_clean
GROUP BY STATE, FIRE_YEAR;

In [0]:
%sql
SELECT *
FROM databricks_7405611871831248.gold.fire_size_state_year
ORDER BY FIRE_YEAR DESC, total_fire_size DESC
LIMIT 50;

## scd 1 part


In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_7405611871831248.gold.dim_fire_scd1 (
  fire_key STRING,
  fire_name STRING,
  state STRING,
  latitude DOUBLE,
  longitude DOUBLE
)
USING DELTA;

## merging to replace

In [0]:
spark.sql("""
MERGE INTO databricks_7405611871831248.gold.dim_fire_scd1 t
USING (
  SELECT DISTINCT
    fire_key,
    FIRE_NAME as fire_name,
    STATE as state,
    LATITUDE as latitude,
    LONGITUDE as longitude
  FROM databricks_7405611871831248.silver.fires_clean
) s
ON t.fire_key = s.fire_key
WHEN MATCHED THEN UPDATE SET
  t.fire_name = s.fire_name,
  t.state = s.state,
  t.latitude = s.latitude,
  t.longitude = s.longitude
WHEN NOT MATCHED THEN INSERT *
""")

## scd 2 part

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_7405611871831248.gold.dim_fire_scd2 (
  fire_key STRING,
  fire_name STRING,
  state STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  is_current BOOLEAN
)
USING DELTA;

In [0]:
spark.sql("""
MERGE INTO databricks_7405611871831248.gold.dim_fire_scd2 t
USING (
  SELECT
    fire_key,
    FIRE_NAME as fire_name,
    STATE as state,
    LATITUDE as latitude,
    LONGITUDE as longitude,
    current_timestamp() as start_date
  FROM databricks_7405611871831248.silver.fires_clean
) s
ON t.fire_key = s.fire_key AND t.is_current = true

WHEN MATCHED AND (
  t.fire_name <> s.fire_name OR
  t.state <> s.state OR
  t.latitude <> s.latitude OR
  t.longitude <> s.longitude
)
THEN UPDATE SET
  t.end_date = current_timestamp(),
  t.is_current = false

WHEN NOT MATCHED THEN
INSERT (
  fire_key,
  fire_name,
  state,
  latitude,
  longitude,
  start_date,
  end_date,
  is_current
)
VALUES (
  s.fire_key,
  s.fire_name,
  s.state,
  s.latitude,
  s.longitude,
  s.start_date,
  NULL,
  true
)
""")

# dimention table---fire dimention

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE databricks_7405611871831248.gold.dim_fire
AS
SELECT DISTINCT
    fire_key,
    FIRE_NAME as fire_name,
    STATE as state,
    LATITUDE as latitude,
    LONGITUDE as longitude
FROM databricks_7405611871831248.silver.fires_clean
""")

# ## Dimentional Table 

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE databricks_7405611871831248.gold.fact_fire_incident
AS
SELECT
    fire_key,
    STATE as state,
    FIRE_SIZE as fire_size
FROM databricks_7405611871831248.silver.fires_clean
""")